# FP 표적 파생 — lin_v2v4 결합학습 (케글, v3b 선형 파이프라인 그대로)
**핵심:** lin_ratio를 *재현*하지 않고 **v3b의 lin_oof 함수를 그대로 복사**해서 씀 → ρ=1.0 보장(재현 오류 불가).
**방법:** `lin_oof(use_ratios=True)` = 기존 v2(lin_ratio) / `lin_oof(use_ratios=True, extra_tr=Ftr, extra_te=Fte)` = v2+FP 결합학습(lin_v2v4).
**판정:** lin_ratio를 lin_v2v4로 **교체** → 블렌드 증분 CI 0 제외.
**출처:** 오류분석 FP 구간(단일이식·저장배아수 +18%p). 트리 미투입(흡수)·선형 결합만.

## 0. 설정 + 유틸 (v3b 셀0 그대로)

In [1]:
import os,glob,re,json,time,warnings
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from scipy.stats import rankdata
import lightgbm as lgb, xgboost as xgb
from catboost import CatBoostClassifier
warnings.filterwarnings("ignore")
DRYRUN=False  # 풀런
SEEDS=(42,) if DRYRUN else (42,7,2024)     # 멤버 생성 시드(그리드는 seed42 기준, 멀티시드는 확인용)
TREE_ITERS=200 if DRYRUN else 1500
def find_csv(n):
    h=[p for p in glob.glob("/kaggle/input/**/*.csv",recursive=True) if os.path.basename(p)==n]; assert h,f"{n} 없음"; return sorted(h,key=len)[0]
train=pd.read_csv(find_csv("train.csv")); test=pd.read_csv(find_csv("test.csv"))
TARGET="임신 성공 여부"; ID_COL="ID"; y=train[TARGET].astype(int).values; N=len(train); base_rate=y.mean()
def NUM(df,c): return pd.to_numeric(df[c],errors="coerce") if c in df else pd.Series(np.nan,index=df.index)
def DIV(num,den): den=den.astype(float); return num.astype(float)/den.where(den>0)
def runr(x): return rankdata(x)/len(x)
COL_PROC="특정 시술 유형"; COL_RSN="배아 생성 주요 이유"
NOMINAL_COLS=["시술 시기 코드","시술 유형","배란 유도 유형","난자 출처","정자 출처"]
OCC=["총 시술 횟수","클리닉 내 총 시술 횟수","IVF 시술 횟수","DI 시술 횟수","총 임신 횟수","IVF 임신 횟수","DI 임신 횟수","총 출산 횟수","IVF 출산 횟수","DI 출산 횟수"]
AGE_MAPS={"시술 당시 나이":{"만18-34세":0,"만35-37세":1,"만38-39세":2,"만40-42세":3,"만43-44세":4,"만45-50세":5,"알 수 없음":-1},
 "난자 기증자 나이":{"만20세 이하":0,"만21-25세":1,"만26-30세":2,"만31-35세":3,"만36-40세":4,"만41-45세":5,"알 수 없음":-1},
 "정자 기증자 나이":{"만20세 이하":0,"만21-25세":1,"만26-30세":2,"만31-35세":3,"만36-40세":4,"만41-45세":5,"알 수 없음":-1}}
CMAP={"0회":0,"1회":1,"2회":2,"3회":3,"4회":4,"5회":5,"6회 이상":6}
_tp=lambda s: [] if pd.isna(s) else [t.strip() for t in re.split(r"[/:]",str(s)) if t.strip()]
print("train",train.shape,"| base_rate=%.4f | DRYRUN=%s | SEEDS=%s | TREE_ITERS=%d"%(base_rate,DRYRUN,SEEDS,TREE_ITERS))

train (256351, 69) | base_rate=0.2583 | DRYRUN=False | SEEDS=(42, 7, 2024) | TREE_ITERS=1500


## 1. 트리 인코더 + 비율 빌더 (v3b 셀1 그대로 — build_lin_ratios 포함)

In [2]:
def fit_tree(tr):
    st={}; ig={TARGET,ID_COL}
    st["dead"]=[c for c in tr.columns if c not in ig and tr[c].nunique(dropna=True)<=1]
    st["sparse"]=[c for c in tr.columns if c not in ig and c not in st["dead"] and tr[c].isna().mean()>0.98]
    st["lc"]={c:pd.Index(tr[c].astype("category").cat.categories) for c in NOMINAL_COLS if c in tr}
    st["pv"]=sorted({t for L in tr[COL_PROC].apply(_tp) for t in L}); return st
def tf_tree(df,st):
    df=df.copy()
    if TARGET in df: df=df.drop(columns=[TARGET])
    df["is_DI"]=(df["시술 유형"]=="DI").astype(int)
    df=df.drop(columns=[c for c in st["dead"] if c in df.columns])
    for c in st["sparse"]:
        if c in df: df[f"{c}_있음"]=df[c].notna().astype(int); df=df.drop(columns=[c])
    for c in OCC:
        if c in df: df[c]=df[c].astype(object).map(CMAP)
    for c,m in AGE_MAPS.items():
        if c in df: df[c]=df[c].astype(object).map(m)
    cats=[]
    for c,cc in st["lc"].items():
        if c in df: df[c]=pd.Categorical(df[c],categories=cc).codes.astype("int32"); cats.append(c)
    ts=df[COL_PROC].apply(_tp)
    for v in st["pv"]: df[f"proc_{v}"]=ts.apply(lambda L,v=v:int(v in L))
    df=df.drop(columns=[c for c in [COL_PROC,COL_RSN,ID_COL] if c in df.columns])
    obj=[c for c in df.columns if df[c].dtype==object]
    if obj: df=df.drop(columns=obj)
    for c in cats: df[c]=df[c].fillna(-1).astype("int32")
    return df,[c for c in cats if c in df.columns]
st=fit_tree(train); Xb,CATF=tf_tree(train,st); Xb_te,_=tf_tree(test,st); Xb_te=Xb_te.reindex(columns=Xb.columns)
# ── v2 게이팅 정본 (3tree-fe-062202와 동일)
def masks(df):
    return {"신선":NUM(df,"신선 배아 사용 여부")==1,"동결":NUM(df,"동결 배아 사용 여부")==1,
            "기증배아":NUM(df,"기증 배아 사용 여부")==1,"DI":(df["시술 유형"].astype(str)=="DI"),
            "ICSI":NUM(df,"미세주입된 난자 수")>0,
            "본인난자":df["난자 출처"].astype(str)=="본인 제공","기증난자":df["난자 출처"].astype(str)=="기증 제공"}
def v1_ratios(df):
    return {"P1":DIV(NUM(df,"총 생성 배아 수"),NUM(df,"혼합된 난자 수")),
            "P2":DIV(NUM(df,"미세주입에서 생성된 배아 수"),NUM(df,"미세주입된 난자 수")),
            "P6":DIV(NUM(df,"총 생성 배아 수"),NUM(df,"수집된 신선 난자 수")),
            "L3":NUM(df,"배아 이식 경과일")-NUM(df,"난자 혼합 경과일"),
            "L6":DIV(NUM(df,"총 생성 배아 수"),NUM(df,"미세주입된 난자 수"))}
def build_v2_gated(df):
    Mk=masks(df); r=v1_ratios(df); F={}
    F["g신선_수정률"]=r["P1"].where(Mk["신선"]); F["gICSI_수정효율"]=r["P2"].where(Mk["ICSI"])
    F["g본인_난자수율"]=r["P6"].where(Mk["본인난자"]); F["g기증_난자수율"]=r["P6"].where(Mk["기증난자"])
    F["g신선_배양일수"]=r["L3"].where(Mk["신선"])
    F["FZ1_동결해동이식간격"]=(NUM(df,"배아 이식 경과일")-NUM(df,"배아 해동 경과일")).where(Mk["동결"])
    F["FZ2_해동이식률"]=DIV(NUM(df,"이식된 배아 수"),NUM(df,"해동된 배아 수"))
    F["FZ3_해동난자수율"]=DIV(NUM(df,"총 생성 배아 수"),NUM(df,"해동 난자 수"))
    F["PG1_PGT강도"]=NUM(df,"착상 전 유전 검사 사용 여부").fillna(0)+NUM(df,"착상 전 유전 진단 사용 여부").fillna(0)
    F["PG2_PGT분류"]=NUM(df,"PGD 시술 여부").fillna(0)+NUM(df,"PGS 시술 여부").fillna(0)
    F["ST1_자극"]=NUM(df,"배란 자극 여부").fillna(0)
    return pd.DataFrame(F,index=df.index)
V2tr=build_v2_gated(train); V2te=build_v2_gated(test)
# ── lin 비율 정본 (lr-fe와 동일)
def build_lin_ratios(df):
    Mk=masks(df); F={}
    P1=DIV(NUM(df,"총 생성 배아 수"),NUM(df,"혼합된 난자 수")); P2=DIV(NUM(df,"미세주입에서 생성된 배아 수"),NUM(df,"미세주입된 난자 수"))
    P6=DIV(NUM(df,"총 생성 배아 수"),NUM(df,"수집된 신선 난자 수")); L3=NUM(df,"배아 이식 경과일")-NUM(df,"난자 혼합 경과일")
    F["P1_수정률"]=P1; F["P2_ICSI수정"]=P2; F["P6_난자수율"]=P6; F["L3_배양일수"]=L3
    F["P3_활용률"]=DIV(NUM(df,"이식된 배아 수")+NUM(df,"저장된 배아 수"),NUM(df,"총 생성 배아 수")); F["P5_저장률"]=DIV(NUM(df,"저장된 배아 수"),NUM(df,"총 생성 배아 수"))
    F["L6_배아perICSI난자"]=DIV(NUM(df,"총 생성 배아 수"),NUM(df,"미세주입된 난자 수")); F["N6_난자감모"]=DIV(NUM(df,"수집된 신선 난자 수")-NUM(df,"혼합된 난자 수"),NUM(df,"수집된 신선 난자 수"))
    F["g신선_수정률"]=P1.where(Mk["신선"]); F["gICSI_수정효율"]=P2.where(Mk["ICSI"]); F["g본인_수율"]=P6.where(Mk["본인난자"]); F["g기증_수율"]=P6.where(Mk["기증난자"]); F["g신선_배양일수"]=L3.where(Mk["신선"])
    F["FZ1_동결해동이식간격"]=(NUM(df,"배아 이식 경과일")-NUM(df,"배아 해동 경과일")).where(Mk["동결"]); F["FZ2_해동이식률"]=DIV(NUM(df,"이식된 배아 수"),NUM(df,"해동된 배아 수"))
    return pd.DataFrame(F,index=df.index)
RATtr=build_lin_ratios(train); RATte=build_lin_ratios(test)
# ── 신규 v3 파생
def build_new_derived(df):
    F={}
    tx =NUM(df,"이식된 배아 수").fillna(0); sto=NUM(df,"저장된 배아 수").fillna(0); emb=NUM(df,"총 생성 배아 수").fillna(0)
    ses=(df["단일 배아 이식 여부"]==1).values
    F["EL_set_type"]=np.where(~ses,0,np.where(sto.values>0,2,1)).astype("int8")
    F["FA_no_transfer"]=(tx==0).astype("int8").values
    is_current=df[COL_RSN].astype(str).str.contains("현재 시술용",na=False).values
    F["FA_nontransfer_reason"]=(~is_current).astype("int8")
    # [C3 폐기: 혼합⊇미세 정의오류]
    return pd.DataFrame(F,index=df.index)
def expand_lin(D):
    E=pd.DataFrame(index=D.index)
    E["EL_forced"]=(D["EL_set_type"]==1).astype("int8"); E["EL_elective"]=(D["EL_set_type"]==2).astype("int8")
    E["FA_no_transfer"]=D["FA_no_transfer"]; E["FA_nontransfer_reason"]=D["FA_nontransfer_reason"]
    return E
Dtr=build_new_derived(train); Dte=build_new_derived(test); Etr=expand_lin(Dtr); Ete=expand_lin(Dte)
CAND={"tree":{"C1":["EL_set_type"],"C2":["FA_no_transfer","FA_nontransfer_reason"]},
      "lin" :{"C1":["EL_forced","EL_elective"],"C2":["FA_no_transfer","FA_nontransfer_reason"]}}
ALL_CANDS=["C1","C2"]   # C3 폐기(혼합⊇미세 정의오류·mixed-route 4%)
def cand_cols(member,cs): return [c for cc in cs for c in CAND[member][cc]]
assert np.array_equal(build_new_derived(train.head(300)).fillna(-9).values, Dtr.head(300).fillna(-9).values), "행단위 독립성 위반!"
print("tf_tree",Xb.shape,"| v2게이팅",V2tr.shape[1],"| lin비율",RATtr.shape[1],"| ✅ 누수안전")

tf_tree (256351, 72) | v2게이팅 11 | lin비율 15 | ✅ 누수안전


## 2. 선형 학습기 (v3b 셀3 그대로 — lin_oof/te_fit/base_num)

In [3]:
LGP=dict(objective="binary",metric="auc",verbose=-1,learning_rate=0.05,num_leaves=63,feature_fraction=0.8,bagging_fraction=0.8,bagging_freq=1,min_child_samples=50)
XGP=dict(objective="binary:logistic",eval_metric="auc",tree_method="hist",learning_rate=0.05,max_depth=6,subsample=0.8,colsample_bytree=0.8)
def fit_one(kind,Xt,yt,Xv,yv,catf,seed):
    if kind=="lgb": return lgb.train(dict(LGP,seed=seed),lgb.Dataset(Xt,yt,categorical_feature=catf),TREE_ITERS,valid_sets=[lgb.Dataset(Xv,yv,categorical_feature=catf)],callbacks=[lgb.early_stopping(80,verbose=False),lgb.log_evaluation(0)])
    if kind=="xgb": return xgb.train(dict(XGP,seed=seed),xgb.DMatrix(Xt.values,label=yt),TREE_ITERS,evals=[(xgb.DMatrix(Xv.values,label=yv),"v")],early_stopping_rounds=80,verbose_eval=False)
    m=CatBoostClassifier(iterations=TREE_ITERS,learning_rate=0.05,depth=6,verbose=0,random_seed=seed,early_stopping_rounds=80); m.fit(Xt,yt,eval_set=(Xv,yv),cat_features=catf); return m
def pred_one(kind,m,X):
    if kind=="lgb": return m.predict(X)
    if kind=="xgb": return m.predict(xgb.DMatrix(X.values),iteration_range=(0,m.best_iteration+1))
    return m.predict_proba(X)[:,1]
def tree_oof(extra_tr=None,extra_te=None,seed=42,kinds=("lgb","xgb","cat"),full_test=False):
    X=Xb if extra_tr is None else pd.concat([Xb.reset_index(drop=True),extra_tr.reset_index(drop=True)],axis=1)
    Xte=Xb_te if extra_te is None else pd.concat([Xb_te.reset_index(drop=True),extra_te.reset_index(drop=True)],axis=1)
    folds=list(StratifiedKFold(5,shuffle=True,random_state=seed).split(X,y)); catf=[c for c in CATF if c in X.columns]; out={}
    for kind in kinds:
        o=np.zeros(N); tt=np.zeros(len(Xte)) if full_test else None
        for tri,vai in folds:
            m=fit_one(kind,X.iloc[tri],y[tri],X.iloc[vai],y[vai],catf,seed); o[vai]=pred_one(kind,m,X.iloc[vai])
            if full_test: tt+=pred_one(kind,m,Xte)/len(folds)
        out[kind]=(o,tt)
    return out
base_num=Xb.drop(columns=CATF); base_num_te=Xb_te.drop(columns=CATF); TE_COLS=NOMINAL_COLS+[COL_PROC,COL_RSN]
def te_fit(cat,yy,sm=20):
    g=pd.DataFrame({"c":cat.values,"y":yy}).groupby("c")["y"].agg(["mean","count"]); pr=float(yy.mean())
    return ((g["mean"]*g["count"]+pr*sm)/(g["count"]+sm)),pr
def lin_oof(use_ratios=False,extra_tr=None,extra_te=None,seed=42,full_test=False):
    oof=np.zeros(N); tst=np.zeros(len(test)) if full_test else None
    for tri,vai in StratifiedKFold(5,shuffle=True,random_state=seed).split(base_num,y):
        Xt=base_num.iloc[tri].copy(); Xv=base_num.iloc[vai].copy(); Xte=base_num_te.copy() if full_test else None
        for c in TE_COLS:
            enc,pr=te_fit(train[c].astype(str).iloc[tri],y[tri])
            Xt[f"te_{c}"]=train[c].astype(str).iloc[tri].map(enc).fillna(pr).values
            Xv[f"te_{c}"]=train[c].astype(str).iloc[vai].map(enc).fillna(pr).values
            if full_test: Xte[f"te_{c}"]=test[c].astype(str).map(enc).fillna(pr).values
        add_tr=[]; add_te=[]
        if use_ratios: add_tr.append(RATtr); add_te.append(RATte)
        if extra_tr is not None: add_tr.append(extra_tr); add_te.append(extra_te)
        if add_tr:
            Xt=pd.concat([Xt.reset_index(drop=True)]+[a.iloc[tri].reset_index(drop=True) for a in add_tr],axis=1)
            Xv=pd.concat([Xv.reset_index(drop=True)]+[a.iloc[vai].reset_index(drop=True) for a in add_tr],axis=1)
            if full_test: Xte=pd.concat([Xte.reset_index(drop=True)]+[a.reset_index(drop=True) for a in add_te],axis=1)
        med=Xt.median(); Xt=Xt.fillna(med); Xv=Xv.fillna(med)
        sc=StandardScaler().fit(Xt); m=LogisticRegression(max_iter=2000,C=0.5).fit(sc.transform(Xt),y[tri])
        oof[vai]=m.predict_proba(sc.transform(Xv))[:,1]
        if full_test: tst+=m.predict_proba(sc.transform(Xte.fillna(med)))[:,1]/5
    return (oof,tst) if full_test else oof
print("학습기 준비")

학습기 준비


## 3. FP 표적 파생 (extra로 주입) + lin_ratio / lin_v2v4 생성

In [4]:
def build_fp_features(df):   # ★ FP 표적(v4). C1 이진이 버린 코호트 내부 해상도(서수).
    F={}
    s=(df["단일 배아 이식 여부"]==1).astype(float).values
    sto=NUM(df,"저장된 배아 수").fillna(0).values; emb=NUM(df,"총 생성 배아 수").fillna(0).values
    fresh=NUM(df,"수집된 신선 난자 수"); eff=np.where(fresh>0, emb/fresh.replace(0,np.nan), np.nan)
    tx=NUM(df,"이식된 배아 수").fillna(0).values
    age=df["시술 당시 나이"].map({"만18-34세":0,"만35-37세":1,"만38-39세":2,"만40-42세":3,"만43-44세":4,"만45-50세":5,"알 수 없음":np.nan}).values
    F["F1_단일x저장수"]=s*sto
    F["F2_단일x난자효율"]=s*np.nan_to_num(eff,nan=0)
    F["F3_단일x활용률"]=s*np.nan_to_num(np.where(emb>0,(tx+sto)/emb,0),nan=0)
    F["F4_단일x나이"]=s*np.nan_to_num(age,nan=2)
    return pd.DataFrame(F,index=df.index)
Ftr=build_fp_features(train); Fte=build_fp_features(test)

# ★ 동일 함수로 둘 다 생성 — 재현 오류 불가
oof_lin_ratio_local,            test_lin_ratio_local = lin_oof(use_ratios=True, seed=42, full_test=True)
oof_lin_v2v4,  test_lin_v2v4 = lin_oof(use_ratios=True, extra_tr=Ftr, extra_te=Fte, seed=42, full_test=True)
print(f"lin_ratio(자체생성) AUC={roc_auc_score(y,oof_lin_ratio_local):.5f}")
print(f"lin_v2v4(결합)      AUC={roc_auc_score(y,oof_lin_v2v4):.5f} (Δ vs v2 {roc_auc_score(y,oof_lin_v2v4)-roc_auc_score(y,oof_lin_ratio_local):+.5f})")
# 외부 저장 lin_ratio와 정합 확인(있으면)
try:
    _ext=pd.read_csv(find_csv("oof_lin_ratio.csv"))["oof_lin_ratio"].values
    if len(_ext)==len(y):
        rho=np.corrcoef(runr(oof_lin_ratio_local),runr(_ext))[0,1]
        print(f"  자체 lin_ratio vs 외부 oof_lin_ratio.csv ρ={rho:.4f} (≈1.0이면 동일 파이프라인 확인)")
except Exception as e: print("  외부 lin_ratio 비교 스킵:",e)

lin_ratio(자체생성) AUC=0.72029
lin_v2v4(결합)      AUC=0.72053 (Δ vs v2 +0.00024)
  자체 lin_ratio vs 외부 oof_lin_ratio.csv ρ=1.0000 (≈1.0이면 동일 파이프라인 확인)


## 4. 블렌드 증분 — lin_ratio → lin_v2v4 교체

In [5]:
def _load(n,col):
    d=pd.read_csv(find_csv(n)); return d[col].values
MEMBERS_FIXED={"lgb":runr(_load("oof_v2v3_trees.csv","oof_v2v3_lgb")),
               "xgb":runr(_load("oof_v3_trees.csv","oof_v3_xgb")),
               "cat":runr(_load("oof_v2v3_trees.csv","oof_v2v3_cat")),
               "nn": runr(_load("oof_nn.csv","oof_nn"))}
def hill_blend(lin):
    d={**MEMBERS_FIXED,"lin":runr(lin)}; nm=list(d); s0={k:roc_auc_score(y,d[k]) for k in nm}
    b=max(s0,key=s0.get); ens=[b]; s=d[b].copy(); best=(list(ens),s0[b])
    for _ in range(150):
        cb,ca=None,-1
        for k in nm:
            a=roc_auc_score(y,(s+d[k])/(len(ens)+1))
            if a>ca: ca,cb=a,k
        ens.append(cb); s=s+d[cb]
        if ca>best[1]: best=(list(ens),ca)
    from collections import Counter; cnt=Counter(best[0]); w={k:cnt.get(k,0)/len(best[0]) for k in nm}
    return sum(w[k]*d[k] for k in nm), w
b0,w0=hill_blend(oof_lin_ratio_local)        # lin=v2
b1,w1=hill_blend(oof_lin_v2v4)               # lin=v2v4
a0=roc_auc_score(y,b0); a1=roc_auc_score(y,b1)
print(f"블렌드 lin=v2   {a0:.5f} | lin 가중치 {w0.get('lin',0):.3f}")
print(f"블렌드 lin=v2v4 {a1:.5f} | lin 가중치 {w1.get('lin',0):.3f}")
rng=np.random.default_rng(0); ds=np.empty(2000)
for i in range(2000):
    ix=rng.integers(0,len(y),len(y)); ds[i]=roc_auc_score(y[ix],b1[ix])-roc_auc_score(y[ix],b0[ix])
lo,hi=np.percentile(ds,[2.5,97.5])
print(f"증분(v2→v2v4 교체) {a1-a0:+.5f} CI[{lo:+.5f},{hi:+.5f}]", "→ 채택후보" if lo>0 else "→ 트리 흡수(닫음)")

블렌드 lin=v2   0.74067 | lin 가중치 0.029
블렌드 lin=v2v4 0.74067 | lin 가중치 0.026
증분(v2→v2v4 교체) -0.00000 CI[-0.00001,+0.00000] → 트리 흡수(닫음)


## 5. 멀티시드 + 저장 (증분 통과 시 그리드용)

In [6]:
if lo>0:
    incs=[]
    for sd in (42,7,2024):
        lr_s=lin_oof(use_ratios=True,seed=sd)
        lv_s=lin_oof(use_ratios=True,extra_tr=Ftr,extra_te=Fte,seed=sd)
        b0s,_=hill_blend(lr_s); b1s,_=hill_blend(lv_s); incs.append(roc_auc_score(y,b1s)-roc_auc_score(y,b0s))
    import numpy as _np; print("멀티시드 증분:",[round(i,5) for i in incs],"평균",round(_np.mean(incs),5),"±",round(_np.std(incs),5))
sfx="_DRYRUN" if DRYRUN else ""
pd.DataFrame({"oof_lin_v2v4":oof_lin_v2v4,"y":y}).to_csv(f"/kaggle/working/oof_lin_v2v4{sfx}.csv",index=False)
pd.DataFrame({"ID":test[ID_COL],"lin_v2v4":test_lin_v2v4}).to_csv(f"/kaggle/working/test_lin_v2v4{sfx}.csv",index=False)
print(f"💾 oof_lin_v2v4{sfx}.csv / test_lin_v2v4{sfx}.csv — 그리드 lin 뷰에 v2v4 추가(lin_ratio 교체 후보)")

💾 oof_lin_v2v4.csv / test_lin_v2v4.csv — 그리드 lin 뷰에 v2v4 추가(lin_ratio 교체 후보)
